In [ ]:
!pip install -q numpy==2.4.4 2>/dev/null
!pip install -q onnx==1.21.0 2>/dev/null
!pip install -q onnxruntime==1.24.4 2>/dev/null
!pip install -q onnx-tool==1.0.1 2>/dev/null
!pip install -q jax2onnx==0.13.0 2>/dev/null
!pip install -q matplotlib 2>/dev/null

In [ ]:
# ── Cell 1: Imports ve yardımcı fonksiyonlar ───────────────────────────────
import os, re, json, shutil, math, copy
from collections import Counter

import numpy as np
import pandas as pd
import jax.numpy as jnp
import jax
import onnx
import onnx.numpy_helper
from jax2onnx import to_onnx
import onnxruntime
import kaggle_benchmarks as kbench

import sys
sys.path.append("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils")
from neurogolf_utils import (
    verify_subset, score_network, run_network,
    load_examples, convert_to_numpy, convert_from_numpy, verify_network,
)

if not hasattr(onnxruntime, "ONNXRuntimeError"):
    onnxruntime.ONNXRuntimeError = Exception

print(f"Devices: {jax.devices()}")


# ── Kod derleme yardımcıları ────────────────────────────────────────────────

def extract_python_code(text: str) -> str:
    if not isinstance(text, str):
        return ""
    match = re.search(r'```python(.*?)```', text, re.DOTALL)
    return match.group(1).strip() if match else text.replace('```', '').strip()


def compile_jax_to_onnx(code_str: str, onnx_filepath: str) -> tuple[bool, str]:
    namespace = {}
    try:
        exec(code_str, globals(), namespace)
        solve_func = namespace.get('solve')
        if not callable(solve_func):
            return False, "'solve' adında çağrılabilir bir fonksiyon bulunamadı."
        dummy_input = jnp.zeros((1, 10, 30, 30), dtype=jnp.float32)
        to_onnx(solve_func, [dummy_input], return_mode="file", output_path=onnx_filepath)
        return True, ""
    except Exception as e:
        return False, str(e)


def compile_onnx_direct(code_str: str, onnx_filepath: str) -> tuple[bool, str]:
    namespace = {}
    _globals = {**globals(), 'onnx': onnx, 'np': np, 'numpy': np,
                'math': math, 'onnx_helper': onnx.helper,
                'TensorProto': onnx.TensorProto}
    try:
        exec(code_str, _globals, namespace)
        build_func = namespace.get('build_model')
        if not callable(build_func):
            return False, "'build_model' adında çağrılabilir bir fonksiyon bulunamadı."
        model = build_func()
        onnx.save(model, onnx_filepath)
        return True, ""
    except Exception as e:
        return False, str(e)


# ── ONNX graf temizleme ─────────────────────────────────────────────────────

def fix_io_names(model: onnx.ModelProto) -> onnx.ModelProto:
    """
    I/O isimlerini 'input'/'output' olarak düzeltir.
    Semantic node isimlerini korur; boş/duplicate isimlere output tensor adını atar.
    Duplicate initializer'ları kaldırır.
    """
    graph = model.graph
    old_inp = graph.input[0].name
    old_out = graph.output[0].name
    if old_inp != "input":
        graph.input[0].name = "input"
        for node in graph.node:
            for i, n in enumerate(node.input):
                if n == old_inp:
                    node.input[i] = "input"
    if old_out != "output":
        graph.output[0].name = "output"
        for node in graph.node:
            for i, n in enumerate(node.output):
                if n == old_out:
                    node.output[i] = "output"

    # Semantic node isimlerini koru; yoksa output tensor adını fallback olarak kullan
    used_names: set[str] = set()
    for node in graph.node:
        base = node.name if node.name else (node.output[0] if node.output else node.op_type)
        candidate = base
        suffix = 0
        while candidate in used_names:
            suffix += 1
            candidate = f"{base}_{suffix}"
        node.name = candidate
        used_names.add(candidate)

    # Duplicate initializer isimlerini temizle
    seen: set[str] = set()
    dup_indices = [
        i for i, init in enumerate(graph.initializer)
        if init.name in seen or seen.add(init.name)  # type: ignore[func-returns-value]
    ]
    for i in reversed(dup_indices):
        del graph.initializer[i]

    return model


# ── ONNX graf füzyon sistemi ────────────────────────────────────────────────

def _build_consumer_map(graph: onnx.GraphProto) -> dict[str, list]:
    """Her tensör için o tensörü tüketen node listesini döndürür."""
    cmap: dict[str, list] = {}
    for node in graph.node:
        for inp in node.input:
            if inp:
                cmap.setdefault(inp, []).append(node)
    return cmap


def _resolve_static(graph: onnx.GraphProto, tensor_name: str):
    """Statik initializer ya da Constant node'dan numpy dizisi döndürür; bulamazsa None."""
    if not tensor_name:
        return None
    for init in graph.initializer:
        if init.name == tensor_name:
            try:
                return onnx.numpy_helper.to_array(init)
            except Exception:
                return None
    for node in graph.node:
        if node.op_type == "Constant" and node.output and node.output[0] == tensor_name:
            for attr in node.attribute:
                if attr.name == "value":
                    try:
                        return onnx.numpy_helper.to_array(attr.t)
                    except Exception:
                        return None
    return None


def _get_attr_ints(node, name: str):
    for attr in node.attribute:
        if attr.name == name:
            return list(attr.ints)
    return None


def _get_attr_int(node, name: str, default: int = 0) -> int:
    for attr in node.attribute:
        if attr.name == name:
            return int(attr.i)
    return default


def _unique_init_name(graph: onnx.GraphProto, base: str) -> str:
    existing = {init.name for init in graph.initializer}
    existing |= {out for n in graph.node for out in n.output if out}
    name, i = base, 0
    while name in existing:
        i += 1
        name = f"{base}_{i}"
    return name


def _remove_node(graph: onnx.GraphProto, target) -> None:
    for i, node in enumerate(graph.node):
        if node is target:
            del graph.node[i]
            return


def _remove_unused_initializers(graph: onnx.GraphProto) -> None:
    used = {inp for node in graph.node for inp in node.input if inp}
    to_del = [i for i, init in enumerate(graph.initializer) if init.name not in used]
    for i in reversed(to_del):
        del graph.initializer[i]


def _cleanup_value_info(graph: onnx.GraphProto, eliminated: set[str]) -> None:
    to_del = [i for i, vi in enumerate(graph.value_info) if vi.name in eliminated]
    for i in reversed(to_del):
        del graph.value_info[i]


def _fuse_reshape_reshape(graph, cmap, log, elim) -> bool:
    """Reshape(x,s1) → Reshape(t1,s2) ⟶ Reshape(x,s2)"""
    graph_outs = {o.name for o in graph.output}
    for node_a in list(graph.node):
        if node_a.op_type != "Reshape" or not node_a.output:
            continue
        t1 = node_a.output[0]
        if not t1 or t1 in graph_outs:
            continue
        consumers = cmap.get(t1, [])
        if len(consumers) != 1 or consumers[0].op_type != "Reshape":
            continue
        node_b = consumers[0]
        node_b.input[0] = node_a.input[0]
        elim.add(t1)
        _remove_node(graph, node_a)
        log.append(f"Reshape('{node_a.name}') → Reshape('{node_b.name}'): eliminated '{t1}'")
        return True
    return False


def _fuse_transpose_transpose(graph, cmap, log, elim) -> bool:
    """Transpose(x,p1) → Transpose(t1,p2) ⟶ Transpose(x, p1∘p2) ya da silinir."""
    graph_outs = {o.name for o in graph.output}
    for node_a in list(graph.node):
        if node_a.op_type != "Transpose" or not node_a.output:
            continue
        t1 = node_a.output[0]
        if not t1 or t1 in graph_outs:
            continue
        consumers = cmap.get(t1, [])
        if len(consumers) != 1 or consumers[0].op_type != "Transpose":
            continue
        node_b = consumers[0]

        n = 4  # ARC tensörleri her zaman 4 boyutlu
        p1 = _get_attr_ints(node_a, "perm") or list(range(n - 1, -1, -1))
        p2 = _get_attr_ints(node_b, "perm") or list(range(n - 1, -1, -1))
        if len(p1) != len(p2):
            continue
        try:
            composed = [p1[p] for p in p2]
        except IndexError:
            continue

        identity = list(range(len(composed)))
        if composed == identity:
            # İki Transpose birbirini iptal ediyor — her ikisini de sil
            t2 = node_b.output[0] if node_b.output else ""
            if t2 in graph_outs:
                continue
            src = node_a.input[0]
            for node in graph.node:
                for i, inp in enumerate(node.input):
                    if inp == t2:
                        node.input[i] = src
            elim.update({t1, t2})
            _remove_node(graph, node_a)
            _remove_node(graph, node_b)
            log.append(f"Transpose('{node_a.name}') → Transpose('{node_b.name}'): identity → both eliminated")
        else:
            node_b.input[0] = node_a.input[0]
            for attr in node_b.attribute:
                if attr.name == "perm":
                    del attr.ints[:]
                    attr.ints.extend(composed)
                    break
            else:
                new_attr = node_b.attribute.add()
                new_attr.name = "perm"
                new_attr.type = onnx.AttributeProto.INTS
                new_attr.ints.extend(composed)
            elim.add(t1)
            _remove_node(graph, node_a)
            log.append(f"Transpose('{node_a.name}') → Transpose('{node_b.name}'): composed perm={composed}")
        return True
    return False


def _fuse_gather_gather(graph, cmap, log, elim) -> bool:
    """Gather(x,i1,a) → Gather(t1,i2,a) ⟶ Gather(x, i1[i2], a)"""
    graph_outs = {o.name for o in graph.output}
    for node_a in list(graph.node):
        if node_a.op_type != "Gather" or len(node_a.input) < 2 or not node_a.output:
            continue
        t1 = node_a.output[0]
        if not t1 or t1 in graph_outs:
            continue
        consumers = cmap.get(t1, [])
        if len(consumers) != 1 or consumers[0].op_type != "Gather":
            continue
        node_b = consumers[0]
        if len(node_b.input) < 2:
            continue
        if _get_attr_int(node_a, "axis", 0) != _get_attr_int(node_b, "axis", 0):
            continue  # Farklı axis'lerde Gather kompozisyonu geçersiz
        i1 = _resolve_static(graph, node_a.input[1])
        i2 = _resolve_static(graph, node_b.input[1])
        if i1 is None or i2 is None:
            continue
        try:
            fused = i1[i2].astype(np.int64)
        except (IndexError, TypeError):
            continue
        fname = _unique_init_name(graph, f"_fused_{node_a.name}_{node_b.name}_idx")
        graph.initializer.append(onnx.numpy_helper.from_array(fused, name=fname))
        node_b.input[0] = node_a.input[0]
        node_b.input[1] = fname
        elim.add(t1)
        _remove_node(graph, node_a)
        log.append(f"Gather('{node_a.name}') → Gather('{node_b.name}'): composed indices, eliminated '{t1}'")
        return True
    return False


def _fuse_matmul_matmul(graph, cmap, log, elim) -> bool:
    """MatMul(W1,x) → MatMul(W2,t1) ⟶ MatMul(W2@W1, x)  [W1 ve W2 statik]"""
    graph_outs = {o.name for o in graph.output}
    for node_a in list(graph.node):
        if node_a.op_type != "MatMul" or len(node_a.input) < 2 or not node_a.output:
            continue
        t1 = node_a.output[0]
        if not t1 or t1 in graph_outs:
            continue
        consumers = cmap.get(t1, [])
        if len(consumers) != 1 or consumers[0].op_type != "MatMul":
            continue
        node_b = consumers[0]
        if len(node_b.input) < 2 or node_b.input[1] != t1:
            continue  # Sadece MatMul(W2, t1) örüntüsünü destekle
        W1 = _resolve_static(graph, node_a.input[0])
        W2 = _resolve_static(graph, node_b.input[0])
        if W1 is None or W2 is None:
            continue
        try:
            W_fused = (W2 @ W1).astype(np.float32)
        except ValueError:
            continue
        fname = _unique_init_name(graph, f"_fused_{node_a.name}_{node_b.name}_W")
        graph.initializer.append(onnx.numpy_helper.from_array(W_fused, name=fname))
        node_b.input[0] = fname
        node_b.input[1] = node_a.input[1]  # orijinal data tensörü x
        elim.add(t1)
        _remove_node(graph, node_a)
        log.append(f"MatMul('{node_a.name}') → MatMul('{node_b.name}'): precomputed W2@W1, eliminated '{t1}'")
        return True
    return False


def _fuse_mul_mul(graph, cmap, log, elim) -> bool:
    """Mul(x,c1) → Mul(t1,c2) ⟶ Mul(x, c1*c2)  [c1 ve c2 statik sabit]"""
    graph_outs = {o.name for o in graph.output}
    for node_a in list(graph.node):
        if node_a.op_type != "Mul" or len(node_a.input) < 2 or not node_a.output:
            continue
        t1 = node_a.output[0]
        if not t1 or t1 in graph_outs:
            continue
        consumers = cmap.get(t1, [])
        if len(consumers) != 1 or consumers[0].op_type != "Mul":
            continue
        node_b = consumers[0]
        if len(node_b.input) < 2:
            continue
        # node_a'da statik sabiti bul
        c1 = _resolve_static(graph, node_a.input[0])
        data_a = node_a.input[1]
        if c1 is None:
            c1 = _resolve_static(graph, node_a.input[1])
            data_a = node_a.input[0]
        if c1 is None:
            continue
        # node_b'de t1 olmayan girdiyi bul
        c2_name = node_b.input[0] if node_b.input[1] == t1 else node_b.input[1]
        c2 = _resolve_static(graph, c2_name)
        if c2 is None:
            continue
        try:
            c_fused = c1 * c2
        except (ValueError, TypeError):
            continue
        fname = _unique_init_name(graph, f"_fused_{node_a.name}_{node_b.name}_scale")
        graph.initializer.append(onnx.numpy_helper.from_array(c_fused.astype(c_fused.dtype), name=fname))
        node_b.input[0] = data_a
        node_b.input[1] = fname
        elim.add(t1)
        _remove_node(graph, node_a)
        log.append(f"Mul('{node_a.name}') → Mul('{node_b.name}'): composed scale, eliminated '{t1}'")
        return True
    return False


def fuse_ops(model: onnx.ModelProto) -> tuple[onnx.ModelProto, list[str]]:
    """
    Algebraik graf füzyonu: Reshape→Reshape, Transpose→Transpose, Gather→Gather,
    MatMul→MatMul, Mul→Mul zincirlerini tek op'a indirger.
    Yakınsayana kadar yinelemeli çalışır. Doğrulama başarısız olursa orijinali döndürür.
    """
    original = copy.deepcopy(model)
    graph = model.graph
    log: list[str] = []
    elim: set[str] = set()

    rules = [
        _fuse_reshape_reshape,
        _fuse_transpose_transpose,
        _fuse_gather_gather,
        _fuse_matmul_matmul,
        _fuse_mul_mul,
    ]

    changed = True
    while changed:
        cmap = _build_consumer_map(graph)
        changed = False
        for rule in rules:
            if rule(graph, cmap, log, elim):
                changed = True
                break  # consumer_map eskidi — yeni geçişe geç

    if elim:
        _cleanup_value_info(graph, elim)
        _remove_unused_initializers(graph)
        try:
            model = onnx.shape_inference.infer_shapes(model)
            onnx.checker.check_model(model)
        except Exception as exc:
            log.append(f"WARN: post-fusion validation failed ({str(exc)[:120]}) — reverted")
            return original, log

    return model, log

In [ ]:
# ── Cell 2: Gözlemci ──────────────────────────────────────────────────────
class Gozlemci:
    """ARC görevi hakkında LLM'e verilmek üzere yapısal bilgi çıkarır."""

    def analyze(self, examples: dict) -> dict:
        train = examples.get('train', [])
        test  = examples.get('test',  [])
        all_pairs = train + test

        if not all_pairs:
            return {'example_summary': 'No examples found.'}

        # Grid boyutları — list kullan, tuple protobuf'ta serileştirilemiyor
        input_sizes  = [[len(p['input']),  len(p['input'][0])]  for p in all_pairs]
        output_sizes = [[len(p['output']), len(p['output'][0])] for p in all_pairs]

        # Renk paleti
        all_colors: set[int] = set()
        for p in all_pairs:
            for row in p['input']:  all_colors.update(row)
            for row in p['output']: all_colors.update(row)

        # Arka plan rengi (ilk train örneğinin girdisinde en sık tekrarlayan)
        flat = [c for row in all_pairs[0]['input'] for c in row]
        background_color = Counter(flat).most_common(1)[0][0]

        # Boyut değişimi var mı?
        size_changes    = [[o[0]-i[0], o[1]-i[1]] for i, o in zip(input_sizes, output_sizes)]
        has_size_change = any(dc != [0, 0] for dc in size_changes)

        # Basit dönüşüm sezgisi
        is_color_mapping = not has_size_change

        unique_in  = sorted({tuple(s) for s in input_sizes})
        unique_out = sorted({tuple(s) for s in output_sizes})
        summary = (
            f"- Pairs: {len(train)} train, {len(test)} test, "
            f"{len(examples.get('arc-gen', []))} arc-gen\n"
            f"- Input sizes (rows×cols): {unique_in}\n"
            f"- Output sizes: {unique_out}\n"
            f"- Size change between input/output: {'YES' if has_size_change else 'NO (same size)'}\n"
            f"- Colors used (channel indices): {sorted(all_colors)}\n"
            f"- Dominant background color: {background_color}\n"
            f"- Transformation hint: "
            f"{'Likely a color/channel mapping' if is_color_mapping else 'Spatial or structural transformation'}"
        )

        return {
            'input_sizes':      input_sizes,
            'output_sizes':     output_sizes,
            'color_palette':    sorted(all_colors),
            'background_color': int(background_color),
            'has_size_change':  bool(has_size_change),
            'is_color_mapping': bool(is_color_mapping),
            'example_summary':  summary,
        }

In [ ]:
# ── Cell 3: Sentezci ──────────────────────────────────────────────────────
class Sentezci:
    """Gözlemci çıktısı + geri bildirimle LLM için hedefli prompt üretir."""

    _MAX_FEEDBACK_CHARS = 1400

    _STATIC_CONSTRAINTS = (
        "CRITICAL CONSTRAINTS:\n"
        "1. STATIC SHAPES: input is float32 [1,10,30,30]. Output MUST be EXACTLY [1,10,30,30].\n"
        "2. FORBIDDEN OPS:\n"
        "   - NO boolean indexing  → creates banned NonZero/Compress ops\n"
        "   - NO jnp.where, jnp.nonzero, jnp.argwhere\n"
        "   - NO dynamic slices based on data values\n"
        "   - NO Loop / Scan / Unique / Script / Function ONNX ops\n"
        "3. ALGEBRAIC MASKING: for 'if cond then A else B' use: (mask*A) + ((1-mask)*B)\n"
        "4. NO RANK REDUCTION: use grid[:,0:1,:,:] not grid[0]\n"
        "5. BACKGROUND: cells outside the active region must be 0.0 across all 10 channels.\n"
    )

    _JAX_SYSTEM = (
        "You are an ML engineer specializing in JAX-to-ONNX static graph compilation.\n"
        "Write a Python function named `solve(grid)` using JAX only.\n"
        "Allowed: jax.numpy ops, jax.scipy.signal.convolve2d, jnp.roll, static padding/slicing.\n\n"
        "EFFICIENCY HIERARCHY — choose the EARLIEST applicable approach (lower = less memory/compute = higher score):\n"
        "  1. Static channel indexing (compiles to Gather — zero learned weights):\n"
        "       grid[:, [4, 2, 0, 1, 3, 5, 6, 7, 8, 9], :, :]   # channel permutation\n"
        "       jnp.take(grid, jnp.array([...]), axis=1)          # same thing\n"
        "  2. Element-wise tensor ops on specific channels (Mul / Add with scalar mask):\n"
        "       grid * mask_constant   # where mask_constant is a static [1,10,1,1] array\n"
        "  3. Reshape + matmul (compiles to Reshape+MatMul, 100 float params):\n"
        "       flat = grid.reshape(1, 10, -1)  # [1,10,900]\n"
        "       out  = jnp.einsum('bcs,cd->bds', flat, W).reshape(1,10,30,30)\n"
        "  4. Convolution — LAST RESORT ONLY (3-5× memory overhead vs Gather).\n"
        "     Use ONLY if the task requires spatial filtering across pixels.\n\n"
        "NAMING REQUIREMENTS:\n"
        "  - First line: # STRATEGY: <one sentence describing your approach>\n"
        "  - Descriptive variable names: `channel_order`, `color_mask`, `spatial_shift` — not `W`, `m`, `x2`\n"
    )

    _ONNX_DIRECT_SYSTEM = (
        "You are an ML engineer building minimal ONNX graphs directly with onnx.helper.\n"
        "Previous JAX approaches failed or scored too low. Build the graph from scratch.\n\n"
        "Write a function `build_model()` that returns an onnx.ModelProto.\n"
        "ALLOWED OPS (ordered by efficiency — prefer ops at the top):\n"
        "  Gather, Transpose, Reshape, MatMul, Gemm,\n"
        "  Add, Sub, Mul, Relu, Sigmoid,\n"
        "  Concat, Slice (STATIC start/end/axes/steps only), Pad,\n"
        "  Conv  ← LAST RESORT (high memory cost, use only for spatial filtering)\n"
        "Input: 'input' float32 [1,10,30,30]   Output: 'output' float32 [1,10,30,30]\n\n"
        "EFFICIENCY HIERARCHY:\n"
        "  ① Gather(axis=1) — direct channel selection, params = 10×int64 = 80 bytes\n"
        "       Use when output channels are a PERMUTATION or SUBSET of input channels.\n"
        "       indices initializer: int64 array of length 10, e.g. [0,2,1,3,4,5,6,7,8,9]\n"
        "  ② Transpose — free axis reorder (e.g. perm=[0,1,3,2] to swap H and W)\n"
        "  ③ Reshape + MatMul([10,10]) + Reshape — linear channel MIX, 100 float params\n"
        "       input [1,10,30,30] → Reshape [10,900] → MatMul W[10,10] → [10,900] → Reshape [1,10,30,30]\n"
        "  ④ Mul / Add with a static [1,10,1,1] mask — per-channel scale/bias, 10 params\n"
        "  ⑤ Conv 1×1 — only if ①-④ cannot express the transformation\n\n"
        "SEMANTIC NAMING (mandatory):\n"
        "  - name='<role>_<op>'  e.g. 'channel_select_gather', 'color_mix_matmul'\n"
        "  - doc_string='<one sentence>'\n"
        "  - model.metadata_props: key='strategy', value='<your approach>'\n\n"
        "GATHER SKELETON (most efficient for channel mapping tasks):\n"
        "```python\n"
        "import onnx, onnx.helper, onnx.numpy_helper, numpy as np\n\n"
        "def build_model():\n"
        "    # e.g. swap channels 3↔5, rest unchanged\n"
        "    idx = np.array([0, 1, 2, 5, 4, 3, 6, 7, 8, 9], dtype=np.int64)\n"
        "    idx_init = onnx.numpy_helper.from_array(idx, name='channel_order')\n"
        "    x = onnx.helper.make_tensor_value_info('input',  onnx.TensorProto.FLOAT, [1,10,30,30])\n"
        "    y = onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1,10,30,30])\n"
        "    gather = onnx.helper.make_node(\n"
        "        'Gather', ['input', 'channel_order'], ['output'],\n"
        "        name='channel_select_gather',\n"
        "        doc_string='Reorder output channels by selecting source channels in learned order',\n"
        "        axis=1,\n"
        "    )\n"
        "    graph = onnx.helper.make_graph([gather], 'g', [x], [y], [idx_init])\n"
        "    model = onnx.helper.make_model(graph, ir_version=10,\n"
        "                                   opset_imports=[onnx.helper.make_opsetid('', 11)])\n"
        "    meta = model.metadata_props.add()\n"
        "    meta.key = 'strategy'\n"
        "    meta.value = 'Permute color channels using a static Gather index'\n"
        "    return model\n"
        "```\n"
        "MATMUL SKELETON (when channels must be linearly mixed):\n"
        "```python\n"
        "def build_model():\n"
        "    W = np.eye(10, dtype=np.float32)  # fill with actual mixing weights\n"
        "    w_init  = onnx.numpy_helper.from_array(W, name='channel_mix_weights')\n"
        "    s1_init = onnx.numpy_helper.from_array(np.array([10, 900], dtype=np.int64), name='shape_flat')\n"
        "    s2_init = onnx.numpy_helper.from_array(np.array([1, 10, 30, 30], dtype=np.int64), name='shape_out')\n"
        "    x = onnx.helper.make_tensor_value_info('input',  onnx.TensorProto.FLOAT, [1,10,30,30])\n"
        "    y = onnx.helper.make_tensor_value_info('output', onnx.TensorProto.FLOAT, [1,10,30,30])\n"
        "    r1     = onnx.helper.make_node('Reshape', ['input', 'shape_flat'], ['flat'],\n"
        "                                   name='flatten_spatial', doc_string='Flatten H×W into one dim')\n"
        "    mm     = onnx.helper.make_node('MatMul',  ['channel_mix_weights', 'flat'], ['mixed'],\n"
        "                                   name='channel_mix_matmul', doc_string='Mix channels linearly')\n"
        "    r2     = onnx.helper.make_node('Reshape', ['mixed', 'shape_out'], ['output'],\n"
        "                                   name='restore_spatial', doc_string='Restore spatial dims')\n"
        "    graph = onnx.helper.make_graph([r1, mm, r2], 'g', [x], [y],\n"
        "                                   [w_init, s1_init, s2_init])\n"
        "    model = onnx.helper.make_model(graph, ir_version=10,\n"
        "                                   opset_imports=[onnx.helper.make_opsetid('', 11)])\n"
        "    meta = model.metadata_props.add(); meta.key = 'strategy'\n"
        "    meta.value = 'Mix color channels via 10×10 MatMul after flattening spatial dims'\n"
        "    return model\n"
        "```\n"
    )

    @staticmethod
    def _trim_feedback(feedback: str) -> str:
        """Feedback'i LLM prompt bütçesi içinde tutar; generik bölümleri atar."""
        for marker in ("\nOPTIMIZATION SUGGESTIONS:", "\nSPARSITY:"):
            idx = feedback.find(marker)
            if idx != -1:
                feedback = feedback[:idx]

        if len(feedback) <= Sentezci._MAX_FEEDBACK_CHARS:
            return feedback.rstrip()

        sections: list[str] = []
        for pattern in (
            r'(=== COST SUMMARY ===.*?)(?=\nGRAPH STRUCTURE|\Z)',
            r'(GRAPH STRUCTURE.*?)(?=\n\n|\Z)',
            r'(MODEL STRATEGY:.*?\n)',
            r'(\[(?:train|test|arc-gen).*?)(?=\n\n|\Z)',
        ):
            m = re.search(pattern, feedback, re.DOTALL)
            if m:
                s = m.group(1).strip()
                if s and s not in sections:
                    sections.append(s)

        trimmed = "\n\n".join(sections)
        if trimmed:
            return trimmed[:Sentezci._MAX_FEEDBACK_CHARS]
        return feedback[:Sentezci._MAX_FEEDBACK_CHARS] + "\n... (truncated)"

    def build_prompt(
        self,
        task_analysis: dict,
        examples: dict,
        attempt: int,
        feedback: str = "",
        prev_code: str = "",
        strategy: str = "jax",
    ) -> str:
        is_optimization = "COST SUMMARY" in feedback and attempt > 1
        strategy_just_changed = (attempt == 4)

        if strategy == "onnx_direct":
            prompt = self._ONNX_DIRECT_SYSTEM + "\n" + self._STATIC_CONSTRAINTS
        else:
            prompt = self._JAX_SYSTEM + "\n" + self._STATIC_CONSTRAINTS

        prompt += "\nTASK ANALYSIS:\n" + task_analysis.get('example_summary', '') + "\n"

        # Görev tipine göre hedefli yönlendirme
        if task_analysis.get('is_color_mapping'):
            prompt += (
                "\nTASK TYPE: CHANNEL MAPPING (same input/output size — no spatial change)\n"
                "→ FIRST try Gather(axis=1) with a static int64 index array — ZERO learned weights.\n"
                "→ If channels must be linearly mixed: Reshape + MatMul([10,10]) + Reshape — 100 params max.\n"
                "→ Do NOT use Conv — its memory footprint is 3-5× higher than Gather for this task.\n"
            )
        else:
            prompt += (
                "\nTASK TYPE: SPATIAL TRANSFORMATION (size or layout changes detected)\n"
                "→ Prefer Slice + Pad + Concat or Transpose for spatial rearrangement.\n"
                "→ Use Conv only if spatial filtering across pixels is truly required.\n"
            )

        if feedback and attempt > 1:
            trimmed_fb = self._trim_feedback(feedback)
            if is_optimization:
                prompt += (
                    f"\n==[ ATTEMPT {attempt-1}: LOGIC CORRECT BUT TOO EXPENSIVE ]==\n"
                    "Your model passed all examples but the memory+param cost is too high.\n"
                    "GOAL: Express the SAME transformation with cheaper ops (see hierarchy above).\n"
                    f"\n{trimmed_fb}\n"
                )
            else:
                prompt += (
                    f"\n==[ ATTEMPT {attempt-1} FAILED ]==\n"
                    f"{trimmed_fb}\n"
                )
            if prev_code and not strategy_just_changed:
                prompt += f"\nYOUR PREVIOUS CODE:\n```python\n{prev_code}\n```\n"

        prompt += "\nTASK EXAMPLES:\n"
        for i, ex in enumerate(examples.get('train', [])):
            prompt += f"Example {i+1}:\nInput:\n{ex['input']}\nOutput:\n{ex['output']}\n\n"

        if strategy == "onnx_direct":
            prompt += "Output ONLY the raw Python code block starting with `def build_model():`."
        else:
            prompt += "Output ONLY the raw Python code block starting with `# STRATEGY:` then `def solve(grid):`."

        return prompt

In [ ]:
# ── Cell 4: Hakem ─────────────────────────────────────────────────────────
class Hakem:
    """
    ONNX modelini değerlendirir:
    - Mantık doğruluğu (tüm örünek subsetleri)
    - Bellek/parametre maliyeti analizi
    - Semantic node isimleri + doc_string + strateji metadata'sını kullanan detaylı geri bildirim
    - Füzyon fırsatı tespiti (fuse_ops sonrası kalan örüntüler)
    """

    def evaluate(self, model: onnx.ModelProto, examples: dict, session) -> dict:
        result = dict(
            is_correct=False, score=0.0,
            agi_pass=0, agi_fail=0,
            gen_pass=0, gen_fail=0,
            memory=0, params=0,
            feedback="", failure_analysis="", cost_analysis="",
        )

        agi_r, agi_w, _ = verify_subset(
            session, examples.get("train", []) + examples.get("test", [])
        )
        gen_r, gen_w, _ = verify_subset(session, examples.get("arc-gen", []))
        result["agi_pass"], result["agi_fail"] = agi_r, agi_w
        result["gen_pass"], result["gen_fail"] = gen_r, gen_w

        trace_path = session.end_profiling()
        total_wrong = agi_w + gen_w

        if total_wrong > 0:
            result["failure_analysis"] = self._analyze_failures(model, session, examples)
            result["feedback"] = (
                f"LOGIC ERROR: {total_wrong} example(s) failed "
                f"({agi_w} ARC-AGI, {gen_w} ARC-GEN).\n"
                + result["failure_analysis"]
            )
            return result

        memory, params = score_network(model, trace_path)
        if memory is None or params is None:
            result["feedback"] = (
                "Static shape/profile error. "
                "Check for dynamic typing, disallowed ops, or mismatched node names."
            )
            return result

        result["memory"]     = memory
        result["params"]     = params
        result["score"]      = round(max(1.0, 25.0 - math.log(max(1.0, memory + params))), 3)
        result["is_correct"] = True
        result["cost_analysis"] = self._analyze_graph_cost(model, memory, params)
        result["feedback"]      = result["cost_analysis"]
        return result

    # ── Strateji metadata okuyucu ─────────────────────────────────────────
    @staticmethod
    def _get_strategy(model: onnx.ModelProto) -> str:
        for prop in model.metadata_props:
            if prop.key == "strategy":
                return prop.value
        return ""

    # ── Graf yapısı özeti ─────────────────────────────────────────────────
    @staticmethod
    def _graph_overview(model: onnx.ModelProto) -> str:
        lines = ["GRAPH STRUCTURE (node index | op | semantic name | role):"]
        for i, node in enumerate(model.graph.node):
            doc = f" — {node.doc_string}" if node.doc_string else ""
            lines.append(f"  [{i:02d}] {node.op_type:<12} '{node.name}'{doc}")
        return "\n".join(lines)

    # ── Füzyon fırsatı tespiti ────────────────────────────────────────────
    @staticmethod
    def _detect_fusion_opportunities(model: onnx.ModelProto) -> list[str]:
        """
        fuse_ops() sonrası atlanamayan füzyon örüntülerini raporlar.
        İki kategori: (1) birden fazla tüketici, (2) dinamik operand.
        LLM bunları düzelterek füzyon yapılabilir hale getirmelidir.
        """
        graph = model.graph
        cmap = _build_consumer_map(graph)
        graph_outs = {o.name for o in graph.output}
        hints: list[str] = []

        FUSABLE = {
            ("Reshape",   "Reshape"),
            ("Transpose", "Transpose"),
            ("Gather",    "Gather"),
            ("MatMul",    "MatMul"),
            ("Mul",       "Mul"),
        }

        for node_a in graph.node:
            if not node_a.output:
                continue
            t1 = node_a.output[0]
            if not t1 or t1 in graph_outs:
                continue
            consumers = cmap.get(t1, [])
            for node_b in consumers:
                if (node_a.op_type, node_b.op_type) not in FUSABLE:
                    continue
                if len(consumers) > 1:
                    hints.append(
                        f"{node_a.op_type}('{node_a.name}') → {node_b.op_type}('{node_b.name}'): "
                        f"'{t1}' has {len(consumers)} consumers — restructure to allow fusion"
                    )
                else:
                    # Tek tüketici ama füzyon olmadı → dinamik operand
                    hints.append(
                        f"{node_a.op_type}('{node_a.name}') → {node_b.op_type}('{node_b.name}'): "
                        f"dynamic operand blocks fusion — use a static initializer"
                    )
        return hints

    # ── Mantık hatası analizi ─────────────────────────────────────────────
    def _analyze_failures(self, model: onnx.ModelProto, session, examples: dict) -> str:
        lines: list[str] = []
        strategy = self._get_strategy(model)
        if strategy:
            lines.append(f"MODEL STRATEGY: {strategy}")
        lines.append(self._graph_overview(model))
        lines.append("")

        subsets = [
            ("train",   examples.get("train", [])),
            ("test",    examples.get("test",  [])),
            ("arc-gen", examples.get("arc-gen", [])[:10]),
        ]
        for label, pairs in subsets:
            for i, ex in enumerate(pairs):
                bench = convert_to_numpy(ex)
                if bench is None:
                    continue
                try:
                    user_out = run_network(session, bench["input"])
                    if np.array_equal(user_out, bench["output"]):
                        continue
                    diff = bench["output"] - user_out
                    wrong_chs = np.where(np.any(diff != 0, axis=(0, 2, 3)))[0].tolist()
                    lines.append(f"[{label}#{i}] Wrong channels: {wrong_chs}")
                    for ch in wrong_chs[:4]:
                        exp_s = float(bench["output"][0, ch].sum())
                        act_s = float(user_out[0, ch].sum())
                        lines.append(f"  ch{ch}: expected_sum={exp_s:.0f}, actual_sum={act_s:.0f}")
                        suspect = self._find_channel_node(model, ch)
                        if suspect:
                            lines.append(f"  ↳ Likely node: '{suspect}'")
                except Exception as exc:
                    lines.append(f"[{label}#{i}] Runtime error: {str(exc)[:120]}")
                if len(lines) >= 30:
                    lines.append("... (truncated)")
                    return "\n".join(lines)
        return "\n".join(lines) if lines else "Failure pattern unknown."

    @staticmethod
    def _find_channel_node(model: onnx.ModelProto, channel: int) -> str:
        init_map = {init.name: init for init in model.graph.initializer}
        candidate = ""
        for node in model.graph.node:
            if node.op_type != "Conv":
                continue
            for inp in node.input:
                if inp in init_map:
                    shape = list(init_map[inp].dims)
                    if shape and shape[0] > channel:
                        candidate = node.name
        return candidate

    # ── Maliyet analizi ───────────────────────────────────────────────────
    def _analyze_graph_cost(self, model: onnx.ModelProto, memory: int, params: int) -> str:
        total = memory + params
        score = max(1.0, 25.0 - math.log(max(1.0, total)))

        lines = ["=== COST SUMMARY ==="]
        strategy = self._get_strategy(model)
        if strategy:
            lines.append(f"Strategy: {strategy}")

        lines += [
            f"Memory: {memory:,} bytes | Params: {params:,} | Total: {total:,}",
            f"Current score: {score:.3f}/25",
            f"To reach 20pts: total ≤ {int(math.exp(5)):,}  "
            f"| 22pts: ≤ {int(math.exp(3)):,}  "
            f"| 25pts: ≤ 1",
            "",
        ]

        # Per-node parametre dağılımı
        init_map = {init.name: init for init in model.graph.initializer}
        node_costs: list[tuple[int, str, str, str, list[str]]] = []
        for node in model.graph.node:
            n_params, details = 0, []
            for inp in node.input:
                if inp in init_map:
                    shape = list(init_map[inp].dims)
                    cnt   = math.prod(shape) if shape else 0
                    n_params += cnt
                    details.append(f"{inp}:{shape}={cnt}p")
            if n_params > 0:
                node_costs.append((n_params, node.op_type, node.name or "",
                                   node.doc_string or "", details))

        node_costs.sort(reverse=True)
        if node_costs:
            lines.append("TOP EXPENSIVE NODES:")
            for cost, op, node_name, doc, details in node_costs[:5]:
                name_tag = f" '{node_name}'" if node_name else ""
                lines.append(f"  {op}{name_tag}: {', '.join(details)}  →  {cost} params")
                if doc:
                    lines.append(f"    ↳ {doc}")
                if op == "Conv":
                    for d in details:
                        m = re.search(r'\[(\d+),(\d+),(\d+),(\d+)\]', d)
                        if m:
                            co, ci, kh, kw = int(m[1]), int(m[2]), int(m[3]), int(m[4])
                            if kh > 1 or kw > 1:
                                lines.append(
                                    f"    → 1×1 kernel alternative: {co*ci} params "
                                    f"(saves {co*ci*kh*kw - co*ci})"
                                )
            lines.append("")

        lines.append(self._graph_overview(model))
        lines.append("")

        # Seyreklik kontrolü
        total_w, zero_w = 0, 0
        for init in model.graph.initializer:
            try:
                arr = onnx.numpy_helper.to_array(init).ravel()
                total_w += arr.size
                zero_w  += int((arr == 0).sum())
            except Exception:
                pass
        if total_w > 0 and zero_w / total_w > 0.5:
            pct = zero_w / total_w * 100
            lines.append(
                f"SPARSITY: {pct:.0f}% of weights are zero. "
                "Consider eliminating zero channels."
            )

        # Füzyon fırsatları (fuse_ops sonrası kalan örüntüler)
        fusion_hints = self._detect_fusion_opportunities(model)
        if fusion_hints:
            lines.append("\nFUSION OPPORTUNITIES (rewrite to eliminate intermediate tensors):")
            for h in fusion_hints[:5]:
                lines.append(f"  • {h}")

        lines += ["", "OPTIMIZATION SUGGESTIONS:"]
        if params > 1000:
            lines.append(
                "  • This transformation may be expressible with a single 1×1 Conv "
                "(10×10=100 params max) or a Gather (0 float params)."
            )
        if len(list(model.graph.node)) > 3:
            lines.append("  • Multiple nodes detected — try merging into fewer ops.")
        if memory > params * 10:
            lines.append(
                "  • Memory footprint is large relative to params. "
                "Check for large intermediate tensors."
            )
        lines.append("  • Prefer Gather > Transpose > MatMul > Conv (see efficiency hierarchy).")
        return "\n".join(lines)

In [ ]:
# ── Cell 5: TaskSolver ────────────────────────────────────────────────────
class TaskSolver:
    """
    Gözlemci → Sentezci → LLM → Derleyici → Füzyon → Hakem döngüsünü yönetir.
    Tüm ara ve nihai çıktıları disk'e kaydeder.
    """

    _MAX_OUTPUT_TOKENS = 1500

    def __init__(
        self,
        llm,
        min_score: float = 15.0,
        max_attempts: int = 8,
        output_dir: str = "/kaggle/working/results",
    ):
        self.llm          = llm
        self.min_score    = min_score
        self.max_attempts = max_attempts
        self.output_dir   = output_dir
        self.gozlemci     = Gozlemci()
        self.sentezci     = Sentezci()
        self.hakem        = Hakem()

    def _call_llm(self, prompt: str) -> str:
        """LLM'i max_output_tokens ile çağırır; desteklenmiyorsa token limitsiz dener."""
        kwargs = {"max_output_tokens": self._MAX_OUTPUT_TOKENS}
        fn = self.llm.prompt if hasattr(self.llm, 'prompt') else self.llm.generate
        try:
            return fn(prompt, **kwargs)
        except TypeError:
            return fn(prompt)

    def solve(self, task_num: int, examples: dict) -> dict:
        task_id  = f"task{task_num:03d}"
        task_dir = os.path.join(self.output_dir, task_id)
        os.makedirs(task_dir, exist_ok=True)

        task_analysis = self.gozlemci.analyze(examples)
        best_result   = {"is_correct": False, "score": 0.0}
        best_onnx_src = None
        feedback      = ""
        prev_code     = ""

        for attempt in range(1, self.max_attempts + 1):
            strategy  = "jax" if attempt <= 3 else "onnx_direct"
            onnx_path = os.path.join(task_dir, f"attempt_{attempt}.onnx")
            trace_pfx = os.path.join(task_dir, f"attempt_{attempt}_trace")

            # 2. Prompt üret
            prompt = self.sentezci.build_prompt(
                task_analysis, examples, attempt,
                feedback, prev_code, strategy,
            )

            # 3. LLM çağrısı
            try:
                raw_response = self._call_llm(prompt)
            except Exception as exc:
                feedback = f"LLM call failed: {exc}"
                print(f"[{task_id}] Attempt {attempt}: LLM error — {exc}")
                continue

            self._save(task_dir, f"attempt_{attempt}_prompt.txt",   prompt)
            self._save(task_dir, f"attempt_{attempt}_response.txt", raw_response)

            # 4. Kod çıkar
            code      = extract_python_code(raw_response)
            prev_code = code
            self._save(task_dir, f"attempt_{attempt}_code.py", code)

            # 5. ONNX derle
            compile_fn = compile_jax_to_onnx if strategy == "jax" else compile_onnx_direct
            success, err_msg = compile_fn(code, onnx_path)
            if not success:
                feedback = f"Compile Error ({strategy}): {err_msg}"
                print(f"[{task_id}] Attempt {attempt} ({strategy}): compile fail")
                continue

            # 6. I/O isim düzelt + algebraik füzyon uygula + kaydet
            try:
                model = onnx.load(onnx_path)
                model = fix_io_names(model)
                model, fusion_log = fuse_ops(model)
                if fusion_log:
                    self._save(task_dir, f"attempt_{attempt}_fusion.txt", "\n".join(fusion_log))
                    n_fused = sum(1 for l in fusion_log if not l.startswith("WARN"))
                    if n_fused:
                        print(f"[{task_id}] Attempt {attempt}: {n_fused} fusion(s) applied")
                onnx.save(model, onnx_path)
            except Exception as exc:
                feedback = f"ONNX load/fix error: {exc}"
                print(f"[{task_id}] Attempt {attempt}: ONNX fix error — {exc}")
                continue

            # 7. OnnxRuntime oturumu (profiling açık)
            try:
                opts = onnxruntime.SessionOptions()
                opts.enable_profiling         = True
                opts.graph_optimization_level = (
                    onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
                )
                opts.profile_file_prefix = trace_pfx
                session = onnxruntime.InferenceSession(
                    model.SerializeToString(), opts
                )
            except onnxruntime.ONNXRuntimeError as exc:
                feedback = f"ONNX session error: {exc}"
                print(f"[{task_id}] Attempt {attempt}: session error — {exc}")
                continue

            # 8. Hakem değerlendirmesi
            eval_result = self.hakem.evaluate(model, examples, session)
            self._save_json(task_dir, f"attempt_{attempt}_result.json", eval_result)

            print(
                f"[{task_id}] Attempt {attempt} ({strategy}): "
                f"correct={eval_result['is_correct']}, "
                f"score={eval_result['score']}, "
                f"agi={eval_result['agi_pass']}/{eval_result['agi_pass']+eval_result['agi_fail']}, "
                f"gen={eval_result['gen_pass']}/{eval_result['gen_pass']+eval_result['gen_fail']}"
            )

            feedback = eval_result["feedback"]

            if eval_result["score"] > best_result["score"]:
                best_result   = eval_result
                best_onnx_src = onnx_path

            if eval_result["is_correct"] and eval_result["score"] >= self.min_score:
                break

        best_onnx_path = None
        if best_onnx_src:
            best_onnx_path = os.path.join(task_dir, f"{task_id}_best.onnx")
            shutil.copy2(best_onnx_src, best_onnx_path)

        summary = {
            "task_id":       task_id,
            "is_correct":    bool(best_result["is_correct"]),
            "best_score":    float(best_result["score"]),
            "attempts_used": int(attempt),
            "onnx_path":     best_onnx_path or "",
        }
        self._save_json(task_dir, "summary.json", {**summary, "task_analysis": task_analysis})
        return summary

    @staticmethod
    def _save(task_dir: str, filename: str, content: str) -> None:
        with open(os.path.join(task_dir, filename), "w", encoding="utf-8") as f:
            f.write(content)

    @staticmethod
    def _save_json(task_dir: str, filename: str, data) -> None:
        with open(os.path.join(task_dir, filename), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, default=str)

In [ ]:
# ── Cell 6: Çalıştırma (kbench) ───────────────────────────────────────────

# ── Parametreler ──────────────────────────────────────────────────────────
MIN_SCORE    = 20.0   # Bu eşiğe ulaşınca görev tamamlanmış sayılır
MAX_ATTEMPTS = 8      # Görev başına maksimum LLM denemesi
OUTPUT_DIR   = "/kaggle/working/results"
TASK_NUMBERS = [4]    # Çözülecek görev numaraları (1-400)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LLM listesi ────────────────────────────────────────────────────────────
try:
    available_models = [
        kbench.llms['google/gemma-4-31b'],
       #kbench.llms['google/gemini-3.1-pro-preview'],
       #kbench.llms['anthropic/claude-opus-4-7@default'],
       #kbench.llms['openai/gpt-5.5-2026-04-23'],
    ]
except KeyError as e:
    print(f"Model bulunamadı: {e}. Mevcut ilk model kullanılıyor.")
    available_models = list(kbench.llms.values())[:1]

# ── kbench görev sarmalayıcısı ─────────────────────────────────────────────
@kbench.task(name="arc_solver", store_task=False)
def arc_task(llm, task_num: int, examples: dict) -> dict:
    solver = TaskSolver(
        llm=llm,
        min_score=MIN_SCORE,
        max_attempts=MAX_ATTEMPTS,
        output_dir=OUTPUT_DIR,
    )
    return solver.solve(task_num, examples)

# ── Değerlendirme verisi ───────────────────────────────────────────────────
evaluation_data = []
for t_num in TASK_NUMBERS:
    ex = load_examples(t_num)
    if ex:
        evaluation_data.append({"task_num": t_num, "examples": ex})

df      = pd.DataFrame(evaluation_data)
n_total = len(available_models) * len(df)

print(f"Değerlendirme: {len(available_models)} model × {len(df)} görev = {n_total} koşum")
for m in available_models:
    print(f"  • {getattr(m, 'name', str(m))}")

# ── Çalıştır ───────────────────────────────────────────────────────────────
runs = arc_task.evaluate(
    stop_condition=lambda runs: len(runs) == n_total,
    max_attempts=1,
    retry_delay=15,
    llm=available_models,
    evaluation_data=df,
    n_jobs=1,
)

# ── Sonuçlar ───────────────────────────────────────────────────────────────
eval_df = runs.as_dataframe()

if not eval_df.empty and 'result' in eval_df.columns:
    metrics_df = pd.json_normalize(eval_df['result'])
    final_df   = pd.concat(
        [eval_df.drop(columns=['result']), metrics_df.set_index(eval_df.index)],
        axis=1,
    )
    final_df['model_name'] = final_df['llm'].apply(
        lambda obj: getattr(obj, 'name', str(obj))
    )

    show_cols = [c for c in
                 ['model_name', 'task_id', 'is_correct', 'best_score', 'attempts_used']
                 if c in final_df.columns]

    print("\n" + "="*60)
    print("SONUÇLAR")
    print("="*60)
    print(final_df[show_cols].to_string(index=False))

    if 'best_score' in final_df.columns:
        summary = final_df.groupby('model_name').agg(
            Dogruluk=('is_correct', 'mean'),
            Ort_Skor=('best_score', 'mean'),
            Toplam_Skor=('best_score', 'sum'),
        )
        print("\nMODEL PERFORMANS ÖZETİ:")
        print(summary)
else:
    print("Sonuç hesaplanamadı.")